In [ ]:
# ┌───────────────────────────────────────────────────────────────┐
# │     Start: validate_and_move_operation_files(args)            │
# └───────────────────────┬───────────────────────────────────────┘
#                         │
#                         ▼
# ┌───────────────────────────────────────────────────────────────┐
# │ Input Params                                                  │
# │ - source_base_path                                            │
# │ - target_base_path                                            │
# │ - failed_files_base_path                                      │
# │ - namespace                                                   │
# │ - watermark_path                                              │
# │ - default_start_date                                          │
# │ - business_event_table_path                                   │
# │ - imaging_metastore_table_path                                │
# └───────────────────────┬───────────────────────────────────────┘
#                         │
#                         ▼
# ┌───────────────────────────────────────────────────────────────┐
# │ Read watermark table: read_watermark(watermark_path)         │
# └───────────────┬───────────────────────────────────────────────┘
#         ┌───────┴────────┐                        
#         │                │                        
#         ▼                ▼                        
# ┌───────────────────┐   ┌────────────────────────────────────────┐
# │ Not Exists/Error  │   │ Exists                                 │
# └──────────┬────────┘   └───────────────┬────────────────────────┘
#            │                            │
#            ▼                            ▼
# ┌───────────────────────────────────────────────┐   ┌──────────────────────────────────────────────┐
# │ Raise ValueError: Inventory watermark missing │   │ Get Inventory watermark for namespace        │
# │ Ask to run Inventory pipeline                 │   │ process_type='Inventory', status='Completed' │
# └───────────────────────────────────────────────┘   └───────────────┬──────────────────────────────┘
#                                                     │
#                                                     ▼
#                                   ┌──────────────────────────────────────────────┐
#                                   │ Inventory last_processed_date found?         │
#                                   └───────────────┬──────────────────────────────┘
#                                                   │ Yes
#                                                   ▼
#                     ┌───────────────────────────────────────────────────────────┐
#                     │ Set end_date = inventory_last_processed_date - 2 days    │
#                     └───────────────────────┬───────────────────────────────────┘
#                                             │
#                                             ▼
#                   ┌─────────────────────────────────────────────────────────────┐
#                   │        Get 'Operations' watermark for namespace             │
#                   │        process_type='Operations', status='Completed'        │
#                   └───────────────┬─────────────────────────────────────────────┘
#                                   │
#                 ┌─────────────────┴──────────────────┐
#                 │                                    │
#                 ▼                                    ▼
# ┌────────────────────────────────────────────┐   ┌─────────────────────────────────────────────┐
# │ Found operations_last_processed_date       │   │ Not found                                   │
# └───────────────┬────────────────────────────┘   └───────────────┬──────────────────────────────┘
#                 │                                                │
#                 ▼                                                ▼
# ┌────────────────────────────────────────────┐   ┌─────────────────────────────────────────────┐
# │ start_date = operations_date + 1 day       │   │ default_start_date provided?                │
# └────────────────────────────────────────────┘   └───────────────┬──────────────────────────────┘
#                                                     │ Yes                          │ No
#                                                     ▼                              ▼
#                              ┌─────────────────────────────────────────────┐   ┌───────────────────────────────────────────────┐
#                              │ start_date = default_start_date             │   │ Find earliest available date under            │
#                              │ (log informational message)                 │   │ source_base_path/namespace                    │
#                              └───────────────────────┬────────────────────┘    │ get_latest_valid_path(..., None)              │
#                                                      │                         └───────────────┬───────────────────────────────┘
#                                                      │                                         │
#                                                      ▼                                         ▼
#                                 ┌────────────────────────────────────────────┐   ┌────────────────────────────────────────────┐
#                                 │ Continue                                   │   │ Earliest date found?                       │
#                                 └────────────────────────────────────────────┘   └───────────────┬────────────────────────────┘
#                                                                                                  │ Yes                     │ No
#                                                                                                  ▼                         ▼
#                                                                     ┌────────────────────────────────────┐   ┌──────────────────────────────────┐
#                                                                     │ start_date = earliest Y/M/D        │   │ Print: No new data; return       │
#                                                                     └────────────────────────────────────┘   └──────────────────────────────────┘
#
# ┌───────────────────────────────────────────────────────────────┐
# │ Log processing window and compute num_days                    │
# └───────────────┬───────────────────────────────────────────────┘
#                 │
#                 ▼
# ┌────────────────────────────────────────────┐
# │ num_days <= 0 ?                            │
# └───────────────┬────────────────────────────┘
#                 │ Yes
#                 ▼
# ┌────────────────────────────────────────────┐
# │ Print: No new data in window; return       │
# └────────────────────────────────────────────┘
#
#                 │ No
#                 ▼
# ┌───────────────────────────────────────────────────────────────┐
# │ For current_date from start_date to end_date (inclusive):     │
# └───────────────┬───────────────────────────────────────────────┘
#                 │
#                 ▼
# ┌───────────────────────────────────────────────────────────────┐
# │ Build date_str (YYYY/MM/DD)                                   │
# │ Derive source_path, target_path, failed_files_path            │
# └───────────────┬───────────────────────────────────────────────┘
#                 │
#                 ▼
# ┌────────────────────────────────────────────────────────────────────────────────────┐
# │ mssparkutils.fs.exists(source_path)?                                               │
# └───────────────┬────────────────────────────────────────────────────────────────────┘
#         │ Yes                                                             │ No
#         ▼                                                                 ▼
# ┌───────────────────────────────────────────────────────────────┐   ┌────────────────────────────────────────┐
# │ Read NDJSON: read_operation_files_from_path(source_path)      │   │ Print: Source path missing; skip day   │
# │ Transform: _transform_patch_data_to_df(df, namespace)         │   └────────────────────────────────────────┘
# │ Validate: process_incremental_data(..., imaging_metastore)    │
# │ rows = final_df.collect()                                     │
# │ Init: total_success=0, total_failed=0, business_events=[]     │
# └───────────────┬───────────────────────────────────────────────┘
#                 │
#                 ▼
# ┌───────────────────────────────────────────────────────────────┐
# │ For each row in rows:                                         │
# │  - filePath, is_exist                                         │
# │  - filename = basename(filePath)                              │
# │  - if is_exist == 'yes': target_full_path = target_path/name  │
# │    else: target_full_path = failed_path/name; ++failed_count; │
# │          add business event via _initialize_business_event()  │
# │  - if target_full_path not exists → mv(filePath, target_full) │
# │    else → print skip                                          │
# └───────────────┬───────────────────────────────────────────────┘
#                 │
#                 ▼
# ┌───────────────────────────────────────────────────────────────┐
# │ Compute success_count = total - failed                        │
# │ Print daily summary                                           │
# │ merge_watermark(watermark_path, 'Operations', current_date,   │
# │                'Completed', namespace)                        │
# │ write_to_business_event(business_events, business_event_path) │
# └───────────────┬───────────────────────────────────────────────┘
#                 │
#                 ▼
# ┌───────────────────────────────────────────────────────────────┐
# │ Increment current_date += 1 day                               │
# └───────────────────────────────────────────────────────────────┘


In [ ]:
##############################################################

# date configuration for validate and move operation
default_start_date = datetime.strptime("2025/11/03", "%Y/%m/%d").date() # This is only applicable for day-0, watermark will take effect onwards.
    # P1 parameter - Ops processing end date
    # Suggestion: more control -- run from specified date or the parquet watermark?

workspace_name = "Imaging_dicom_checkpoint_2026_02_06"
bronze_lakehouse_name = "healthcare1_msft_bronze"
silver_lakehouse_name = "healthcare1_msft_silver"
admin_lakehouse_name = "healthcare1_msft_admin"

# Replace with parameterized source base path (abfss://<workspace>@<tenant>-onelake.dfs.microsoft.com/<lakehousename>.Lakehouse/Files...)
source_base_path = f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_name}.Lakehouse/Files/Ingest/Imaging/OPERATIONS/"
target_base_path = f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_name}.Lakehouse/Files/Process/Imaging/OPERATIONS/"
failed_files_base_path = f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_name}.Lakehouse/Files/Failed/Imaging/OPERATIONS/"
business_event_table_path = f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{admin_lakehouse_name}.Lakehouse/Tables/BusinessEvents"
imaging_metastore_table_path = f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{silver_lakehouse_name}.Lakehouse/Tables/ImagingMetastore"
watermark_path = f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_name}.Lakehouse/Tables/CustomDicomWatermark"

In [ ]:
def list_dirs(path):
    return [f.name.strip("/") for f in mssparkutils.fs.ls(path) if f.isDir]

def get_latest_valid_path(base_path, watermark_dt):

    years = sorted(list_dirs(base_path), reverse=False)

    for y in years:
        year_path = f"{base_path}/{y}"
        months = sorted(list_dirs(year_path), reverse=False)

        for m in months:
            month_path = f"{year_path}/{m}"
            days = sorted(list_dirs(month_path), reverse=False)

            for d in days:
                folder_date = date(int(y), int(m), int(d))

                if watermark_dt is None:
                    return y, m, d, f"{month_path}/{d}"
                elif folder_date > watermark_dt:
                    return y, m, d, f"{month_path}/{d}"

    return None

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date, lit
from datetime import datetime
from pyspark.sql.types import *

def merge_watermark(watermark_path, process_type, last_processed_date, status, namespace):
    # Schema
    schema = StructType([
        StructField("process_type", StringType(), True),
        StructField("last_processed_date", DateType(), True),
        StructField("status", StringType(), True),
        StructField("namespace", StringType(), True)
    ])

    is_watermark_exist, target_watermark_df = read_watermark(watermark_path)

    # Create the source DataFrame
    import datetime
    data = [
        (process_type, last_processed_date, status, namespace)    
    ]
    source_watermark_df = spark.createDataFrame(data, schema=schema)

    if is_watermark_exist:
        # Load the target table as a DeltaTable
        target = DeltaTable.forPath(spark, watermark_path)

        # Perform MERGE with update + insert
        (
            target.alias("target")
            .merge(
                source_watermark_df.alias("source"),
                "target.process_type = source.process_type AND target.namespace = source.namespace"                
            )
            .whenMatchedUpdate(
                condition="target.status = 'Completed'",   # update only if status is 'complete'
                set={
                    "status": "source.status",
                    "namespace": "source.namespace",                    
                    "last_processed_date": "source.last_processed_date"
                }
            )
            .whenNotMatchedInsert(values={
                "process_type": "source.process_type",
                "last_processed_date": "source.last_processed_date",
                "status": "source.status",
                "namespace": "source.namespace"
            })
            .execute()
        )
    else:    
        source_watermark_df.write.format("delta").mode('overwrite').save(watermark_path)
        print(f"Successfully created watermark table: {watermark_path}")  


In [ ]:
def read_watermark(watermark_path):
    """Reads the watermark date from a Delta table, or returns the default date if the table doesn't exist."""
    from pyspark.sql.functions import col
    try:
        # Check if the watermark directory/table exists
        if mssparkutils.fs.exists(watermark_path):
            # Read the Delta table
            df_watermark = spark.read.format("delta").load(watermark_path)
            if not df_watermark.rdd.isEmpty():
                return True, df_watermark
            else:
                return False, None
        else:
            return False, None

    except Exception as e:
        raise ValueError(f"Error reading watermark: {e}.")

In [ ]:
def get_watermark_date(df_watermark, process_type, status , namespace):
    """Reads the watermark date from a Parquet file, or returns the default date if the file doesn't exist."""
    from pyspark.sql.functions import col
    try:
        # Example: process_type variable already defined
        filtered_df = df_watermark.filter(
            (col("process_type") == process_type) & (col("status") == status) & (col("namespace") == namespace)
        )
        if not filtered_df.isEmpty():
            # Get the first matching row
            row = filtered_df.select("last_processed_date", "status").first()
        
            # Assign to Python variables
            if row:
                last_processed_date = row["last_processed_date"]
                status = row["status"]

                # print("Last Processed Date:", last_processed_date)
                # print("Status:", status)
        
                # Convert the date object (if stored as date/timestamp) or string back to datetime
                if isinstance(last_processed_date, str):
                    # Assuming date is stored as 'YYYY/MM/DD' string
                    return date(datetime.strptime(last_processed_date, "%Y/%m/%d"))
                elif isinstance(last_processed_date, (datetime, type(datetime.now().date()))):
                    # If stored as Date or Timestamp, ensure it's a datetime object
                    return date(last_processed_date.year, last_processed_date.month, last_processed_date.day)
        else:
            return None
                
    except Exception as e:
        raise ValueError(f"Error reading watermark: {e}.No matching row found for process_type = '{process_type}', Please run 'Inventory' pipeline to complete the dicom extrction process")

In [ ]:
def _initialize_business_event(namespace: str, sourceFilePath = None, targetFilePath = None)-> dict:

    import uuid
    from datetime import datetime    
    
    business_events_row = {
        "id": str(uuid.uuid4()),
        "activityName": "imaging_custom_operations_file_movement",        
        "targetTableName": "NA",
        "targetFilePath": targetFilePath,
        "sourceTableName": namespace,
        "sourceLakehouseName": "Bronze",
        "targetLakehouseName": "Bronze",
        "sourceFilePath": sourceFilePath,
        "runId": "NA",
        "severity": "error",
        "eventType": "Operations file movement from ADLS shortcut to Process folder",
        "recordIdentifier": "NA",                # or some unique study/record id
        "recordIdentifierSource": "system",      # e.g. PACS, Metastore, etc.
        "active": True,                          # boolean flag
        "message": "study information is not present in ImagingMetastore",
        "exception": "studyid information is not present in ImagingMetastore",        
        "customDimensions": "{}",                # JSON string or key=value pairs
        "eventDateTime": datetime.utcnow(),      # current timestamp
        "createdDatetime": datetime.utcnow()     # creation timestamp
    }

    return business_events_row            

In [ ]:
#patch file ingestion constants
TAG_STRING = "tag_string"
STUDY_INSTANCE_UID = "studyInstanceUid"
SERIES_INSTANCE_UID = "seriesInstanceUid"
SOP_INSTANCE_UID = "sopInstanceUid"
OPERATION = "operation"
DELETE_OPERATION = "soft-delete"
PATCH_OPERATION = "patch"
TAGS_JSON_COLUMN_NAME = "metadata"
TAGS_METADATA_STRING = "metadata_string"
    

FILE_PATH_COLUMN_NAME = "filePath"
SOURCE_MODIFIED_COLUMN_NAME = "sourceModifiedAt"
SOURCE_SYSTEM_COLUMN_NAME = "sourceSystem"
MSFT_SOURCE_SYSTEM_COLUMN_NAME = "msftSourceSystem"

from pyspark.sql.types import *
# from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType

PATCH_FILE_SCHEMA = StructType([
    StructField("id", StringType(), True),
    StructField("meta", StructType([
        StructField("lastUpdated", StringType(), True)
    ]), True),
    StructField("resourceType", StringType(), True),
    StructField("identifier", ArrayType(StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True),
    ])), True),
    StructField("series", ArrayType(StructType([
        StructField("uid", StringType(), True),
        StructField("instance", ArrayType(StructType([  # Explicitly define instance as an ArrayType of StructType
            StructField("uid", StringType(), True)
        ]), True), True)
    ])), True),
    StructField("extension", ArrayType(StructType([
        StructField("url", StringType(), True),
        StructField("valueString", StringType(), True)
    ])), True)
])

BUSINESS_EVENT_SCHEMA = StructType([
    StructField("id", StringType(), True),
    StructField("activityName", StringType(), True),
    StructField("targetTableName", StringType(), True),
    StructField("targetFilePath", StringType(), True),
    StructField("sourceTableName", StringType(), True),
    StructField("sourceLakehouseName", StringType(), True),
    StructField("targetLakehouseName", StringType(), True),
    StructField("sourceFilePath", StringType(), True),
    StructField("runId", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("eventType", StringType(), True),
    StructField("recordIdentifier", StringType(), True),
    StructField("recordIdentifierSource", StringType(), True),
    StructField("active", BooleanType(), True),
    StructField("message", StringType(), True),
    StructField("exception", StringType(), True),
    StructField("customDimensions", StringType(), True),
    StructField("eventDateTime", TimestampType(), True),
    StructField("createdDatetime", TimestampType(), True)
])

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import filter, col, to_json,from_json, lit, when, size,input_file_name, lower, concat, array, expr,struct,collect_list,row_number, coalesce

def _transform_patch_data_to_df(patch_df: DataFrame, namespace: str) -> DataFrame:
        """
        Transforms a DataFrame containing patch file metadata into a structured DataFrame with extracted fields.
        It also parses JSON metadata and tranform into ImagingDicom table schema.

        Args:
            patch_df (DataFrame): The DataFrame containing the patch file data.
            namespace (str): The namespace/source system name.
            
        Returns:
            DataFrame: The transformed metadata DataFrame.
        """
        # Extracting operation, tags, series_uid, and instance_uid
        df_transformed = patch_df.withColumn(
            OPERATION,
            expr("""
                filter(extension, x -> lower(trim(x.url)) = 'http://healthcaredatasolutions.com/data-extensions/operation')[0].valueString
            """)
        ).withColumn(FILE_PATH_COLUMN_NAME, input_file_name()
        ).withColumn(SOURCE_MODIFIED_COLUMN_NAME,col("meta").getField("lastUpdated").cast("timestamp")                  
        ).withColumn(STUDY_INSTANCE_UID,col("identifier").getItem(0).getField("value")                
        ).withColumn(
            SERIES_INSTANCE_UID,
            when(size(col("series")) > 0, col("series").getItem(
                0).getField("uid")).otherwise(lit(None))
        ).withColumn(
            SOP_INSTANCE_UID,
            when(
                (size(col("series")) > 0) &
                (col("series").getItem(0).getField("instance").isNotNull()) &
                (size(col("series").getItem(0).getField("instance")) > 0),
                col("series").getItem(0).getField(
                    "instance").getItem(0).getField("uid")
            ).otherwise(lit(None))        
        ).withColumn(SOURCE_SYSTEM_COLUMN_NAME,lit(namespace))
               
        return df_transformed

In [ ]:
from delta.tables import DeltaTable
from typing import Any, List, Optional, Union

def find_managed_delta_table_using_path(
     delta_table_path: str
) -> Union[DeltaTable, None]:
    
    delta_result = None

    try:
        delta_result = DeltaTable.forPath(spark, delta_table_path)
    except AnalysisException:
        return delta_result
    return delta_result

In [ ]:
from pyspark.sql.functions import input_file_name,col

def read_operation_files_from_path(sourcePath : str):    
    df = (
        spark.read
            # NDJSON: one JSON object per line, so multiline must be false (default)
            .option("multiline", "false")
            # Read nested folders
            .option("recursiveFileLookup", "true")
            # Only files that match the glob will be read
            .option("pathGlobFilter", "*.ndjson")
            # (Optional) handle corrupt records if schema mismatches occur
            .option("mode", "PERMISSIVE")
            .schema(PATCH_FILE_SCHEMA)   
            .json(sourcePath)
    ).withColumn("source_file", input_file_name())

    return df

In [ ]:
def write_to_business_event(events, business_event_table_path : str) :
    
    from pyspark.sql.types import (
    StructType, StructField,
    StringType, BooleanType, TimestampType
    )

    if not events:
        print("No events to append")
        return
 
    event_df = spark.createDataFrame(events,schema = BUSINESS_EVENT_SCHEMA) 
    (
        event_df
        .write
        .mode("append")
        .format("delta")
        .save(business_event_table_path)
    )

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, when, lit

def process_incremental_data(incremental_df : DataFrame, metastore_table_path : str): 
    incremental_df = incremental_df.select(col("filePath"),SOURCE_SYSTEM_COLUMN_NAME,STUDY_INSTANCE_UID,SERIES_INSTANCE_UID,SOP_INSTANCE_UID)
    
    metastore_df = find_managed_delta_table_using_path(metastore_table_path)
    if metastore_df is not None:
        metastore_df = metastore_df.toDF()
        metastore_df = metastore_df.select(MSFT_SOURCE_SYSTEM_COLUMN_NAME,STUDY_INSTANCE_UID,SERIES_INSTANCE_UID,SOP_INSTANCE_UID)
    else:
        print("Delta table not found at the specified path.")

    # Aliases for readability
    inc  = incremental_df
    meta = metastore_df

    # Your join condition
    cond = (
        (meta[STUDY_INSTANCE_UID] == inc[STUDY_INSTANCE_UID]) &
        (meta[MSFT_SOURCE_SYSTEM_COLUMN_NAME] == inc[SOURCE_SYSTEM_COLUMN_NAME]) &
        (
            (inc[SERIES_INSTANCE_UID].isNull() | (meta[SERIES_INSTANCE_UID] == inc[SERIES_INSTANCE_UID])) &
            (inc[SOP_INSTANCE_UID].isNull()    | (meta[SOP_INSTANCE_UID]    == inc[SOP_INSTANCE_UID]))
        )
    )

    # Perform single left outer join and add flag
    joined_df = (
        inc.join(meta, cond, how="left_outer")
        .select(
            inc["filePath"],
            inc[SOURCE_SYSTEM_COLUMN_NAME],
            inc[STUDY_INSTANCE_UID],
            inc[SERIES_INSTANCE_UID],
            inc[SOP_INSTANCE_UID],
            meta[STUDY_INSTANCE_UID].alias("meta_study_uid"),
            meta[SERIES_INSTANCE_UID].alias("meta_series_uid"),
            meta[SOP_INSTANCE_UID].alias("meta_sop_uid")
        )
        # String flag: "not exist" if ANY right-side key is NULL
        .withColumn(
            "is_exist",
            when(
                col("meta_study_uid").isNull() |
                col("meta_series_uid").isNull() |
                col("meta_sop_uid").isNull(),
                lit("no")
            ).otherwise(lit("yes"))
        ))

    final_df = joined_df.dropDuplicates(["filePath"])
    return final_df

In [ ]:
from datetime import datetime, timedelta, date
from notebookutils import mssparkutils
import os

from pyspark.sql import SparkSession,Row
from pyspark.sql.functions import lit

def validate_and_move_operation_files(source_base_path : str, 
                        target_base_path :str,
                        failed_files_base_path : str,
                        namespace : str, 
                        watermark_path : str,
                        default_start_date : datetime,                        
                        business_event_table_path : str,
                        imaging_metastore_table_path: str):
                        

    # The 'end_date' here is the *hard stop* date for the current run.
    # --- Read Watermark and Set Start Date ---
    print("#"*80)
    print(f"{namespace}")
    print("#"*80)
    is_watermark_exist, watermark_df = read_watermark(watermark_path)
    if is_watermark_exist:        
        inventory_last_processed_date = get_watermark_date(watermark_df, "Inventory", "Completed", namespace)
        print("Watermark : DICOM Inventory files last processed date:", inventory_last_processed_date)
        # print("Status:", status)
        if inventory_last_processed_date is not None:            
            end_date = inventory_last_processed_date - timedelta(days=2)
        else:
            print("No matching row found for process_type = 'Inventory' and status = 'Completed', Please run 'Inventory' pipeline to complete the dicom extraction process")
            return

        source_base_path_ns = os.path.join(source_base_path, namespace)
        operations_last_processed_date = get_watermark_date(watermark_df, "Operations", "Completed", namespace)
        print("Watermark : Opeartions (path/delete) files last processed date:", operations_last_processed_date)

        if operations_last_processed_date is not None:
            start_date = operations_last_processed_date  + timedelta(days=1)
        else:
            if default_start_date is not None:
                start_date = default_start_date
                print("Since the watermark is empty for Opeartions (path/delete) files,processing will begin from the 'default_start_date' :", start_date)
            else:
                result = get_latest_valid_path(source_base_path_ns, None)
                if not result:
                    print("No new data to process based on current watermark and end date.")
                    return            
                year, month, day, _ = result
                start_date = date(int(year), int(month), int(day))
                print("Since the watermark is empty for Opeartions (path/delete) files,processing will begin from the earlier available date :", start_date)
    else:
        raise ValueError(f"No matching row found for process_type = 'Inventory' and status = 'Completed', Please run 'Inventory' pipeline to complete the dicom extrction process")
    
    

    print(f"Processing window: start_date - {start_date.strftime('%Y/%m/%d')} to end_date - {end_date.strftime('%Y/%m/%d')}") 
    num_days = (end_date - start_date).days + 1

    if num_days <= 0:
        print("No new data to process based on current watermark and end date.")    
    else:
        print(f"Processing {num_days} days from {start_date.strftime('%Y/%m/%d')} to {end_date.strftime('%Y/%m/%d')}")
                
        current_date = start_date
        while current_date <= end_date:
            date_str = current_date.strftime("%Y/%m/%d")
            path_without_day = "/".join(date_str.rsplit("/", 1)[0:1])
            
            source_path = os.path.join(source_base_path, namespace, date_str)
            target_path = os.path.join(target_base_path, namespace, date_str)
            failed_files_path = os.path.join(failed_files_base_path, namespace, date_str)
            
            print(f"Processing: {date_str}")
            print(f"Source: {source_path}")
            print(f"Target: {target_path}")
            
            if mssparkutils.fs.exists(source_path):
                df = read_operation_files_from_path(source_path)
                incremental_df = _transform_patch_data_to_df(df, namespace)
                final_df = process_incremental_data(incremental_df, imaging_metastore_table_path)
                
                rows = final_df.collect()  # Collect all rows to driverrows = df.collect()  # Collect
            
                total_success_count = 0
                total_failed_count = 0
                total_count = len(rows)
                business_events = []
                for row in rows:                    
                    file_path = row["filePath"]
                    is_exist_flag = row["is_exist"]

                    filename = os.path.basename(file_path)
                    if is_exist_flag == 'yes':
                        target_full_path = os.path.join(target_path,filename)
                    else:                          
                        target_full_path = os.path.join(failed_files_path,filename)
                        total_failed_count += 1
                        bs_row = _initialize_business_event(namespace, target_full_path, target_full_path)
                        business_events.append(Row(**bs_row))
                        
                    #print(filename)
                    if not mssparkutils.fs.exists(target_full_path):
                        mssparkutils.fs.cp(file_path, target_full_path, True)
                        mssparkutils.fs.rm(file_path)
                    else:
                        print(f"Target file already exists: {target_full_path}, skipping copy.")

                total_success_count = total_count - total_failed_count
                print(f"Total number of rows getting processed:\n"
                            f"\t source_path : {source_path}\n"
                            f"\t target_path : {target_path}\n"
                            f"\t total_count : {len(rows)}\n"
                            f"\t total_success_count : {total_success_count}\n"
                            f"\t total_failed_count : {total_failed_count}")
                # --- Write New Watermark (The crucial step after successful loop) ---
                # The watermark is the LAST successful date processed, which is processing_end_date.
                #write_watermark(watermark_path, watermark_column, current_date)                
                merge_watermark(watermark_path, "Operations", current_date , "Completed", namespace)
                # watermark_path = "abfss://Imaging_dicom_br_30_09@msit-onelake.dfs.fabric.microsoft.com/healthcare1_msft_bronze.Lakehouse/Tables/CustomDicomWatermark"
                write_to_business_event(business_events,business_event_table_path)
            else:
                print(f"Source path does not exist: {source_path}")         
            
            current_date += timedelta(days=1)
            

In [ ]:
def get_lst_namespaces(source_base_path, namespace=None):
    namespaces=None

    if isinstance(namespace, str):
        namespaces = namespace.split(",")
        return namespaces
    
    elif namespace is None or len(namespace) == 0:
        # List all namespaces (folders) under the path
        entries = mssparkutils.fs.ls(source_base_path)
        namespaces = [e.name.replace("/", "") for e in entries if e.isDir]
        return namespaces
    else:
        raise Exception("Invalid namespaces parameter")
    return namespaces

In [ ]:
def validate_and_move_operation_files_for_namespace(namespace_list):
    for namespace in namespace_list:
        validate_and_move_operation_files(source_base_path, 
                                         target_base_path, 
                                         failed_files_base_path, 
                                         namespace, 
                                         watermark_path, 
                                         default_start_date, 
                                         business_event_table_path,
                                         imaging_metastore_table_path)

In [ ]:
str_namespaces=None
namespace_list = get_lst_namespaces(source_base_path, str_namespaces)
print(namespace_list)
validate_and_move_operation_files_for_namespace(namespace_list)

In [ ]:
df = spark.sql("SELECT * FROM healthcare1_msft_admin.BusinessEvents where eventDatetime >= '2026-02-01' LIMIT 1000")
display(df)


In [ ]:
df = spark.sql("SELECT * FROM healthcare1_msft_bronze.CustomDicomWatermark LIMIT 1000")
display(df)